# 🧪 Notebook 03: Model Architecture Prototyping & Comparison

This notebook prototypes and compares different neural network architectures for anomaly detection:
1. **SQLCNN**: 1D Convolutional Neural Network with multi-scale kernel sizes (3, 4, 5).
2. **SQLLSTM**: Bidirectional Long Short-Term Memory Network with multi-layer recurrent units.

We benchmark:
- Parameter counts and model footprint
- Forward pass inference latency on CPU vs GPU
- Validation performance metrics on SQL injection detection

In [ ]:
import os
import sys
import time
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

# Add project root to path
sys.path.append(os.path.abspath('../'))
from src.models.sql_models import SQLCNN, SQLLSTM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load vocabulary to determine embedding input dimension
vocab_path = '../data/processed/sql_word2idx.json'
if os.path.exists(vocab_path):
    with open(vocab_path, 'r', encoding='utf-8') as f:
        word2idx = json.load(f)
    vocab_size = len(word2idx)
else:
    vocab_size = 5000

print(f"Vocabulary size: {vocab_size}")


### 1. Initialize Prototype Architectures and Compare Parameter Counts

In [ ]:
cnn_model = SQLCNN(vocab_size=vocab_size, embed_dim=128, num_filters=100, filter_sizes=[3, 4, 5]).to(device)
lstm_model = SQLLSTM(vocab_size=vocab_size, embed_dim=128, hidden_dim=64, num_layers=2).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

cnn_params = count_parameters(cnn_model)
lstm_params = count_parameters(lstm_model)

print(f"SQLCNN trainable parameters:  {cnn_params:,}")
print(f"SQLLSTM trainable parameters: {lstm_params:,}")

comparison_df = pd.DataFrame({
    'Architecture': ['SQLCNN', 'SQLLSTM'],
    'Parameters': [cnn_params, lstm_params],
    'Model Type': ['1D Multi-Kernel CNN', 'Bidirectional 2-Layer LSTM'],
    'Primary Advantage': ['Fast parallel feature extraction', 'Sequential dependency modeling']
})
display(comparison_df)


### 2. Benchmark Inference Latency

In [ ]:
# Benchmark batch of queries
batch_size = 64
seq_len = 100
dummy_batch = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long).to(device)

def benchmark_latency(model, input_batch, num_runs=100):
    model.eval()
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(input_batch)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    start_time = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(input_batch)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    avg_latency_ms = ((time.time() - start_time) / num_runs) * 1000
    return avg_latency_ms

cnn_latency = benchmark_latency(cnn_model, dummy_batch)
lstm_latency = benchmark_latency(lstm_model, dummy_batch)

print(f"SQLCNN  average inference latency (batch={batch_size}): {cnn_latency:.2f} ms")
print(f"SQLLSTM average inference latency (batch={batch_size}): {lstm_latency:.2f} ms")
print(f"Speedup factor (CNN vs LSTM): {lstm_latency / max(cnn_latency, 1e-6):.2f}x faster")


### 3. Conclusion & Model Selection
- **SQLCNN** provides significantly faster inference throughput and parallel token feature extraction, making it optimal for the real-time active database firewall.
- **SQLLSTM** serves as a strong sequential benchmark, with weights preserved under `notebooks/SQL_LSTM_best.pth`.